In [1]:
import os
import glob
import pandas as pd


#data_source = 'data/backups/results_5'
data_source = 'data/results'
samples = []
for file_name in os.listdir(data_source):
    if file_name.endswith('.csv'):
        samples.append(pd.read_csv(f"{data_source}/{file_name}"))

# Expected Matching Approach - Interactive Dashboard

## Overview

This dashboard visualizes the results of the comparative simulation study conducted in Notebook 2. It allows you to interactively explore how different matching algorithms perform across various metrics, conditions, and scenarios.

## Quick Start Guide

### What You'll Find Here:
1. **Utility Loss & Robustness**: How well does each system qualify the "right" teams?
2. **Top 8 Precision**: What percentage of the true top 8 teams actually qualify?
3. **Fairness**: How likely are the best teams to progress throughout the tournament?
4. **Balance**: Are games competitive or one-sided?
5. **Last Round Metrics**: Special focus on the critical final qualification round
6. **Frustration**: How often does a visible injustice happen?

### Double Perspective

Each presented metrics are computed using an input ranking. 

- **LEVEL** Perspectve uses the ground truth ranking from the model used to compute game outcome. This adresses *True* tournament quality as it explores the performances of a pairing system regarding the ability of teams to win games.
- **SEED** Perspective uses the seedings passed to the tournament for pairings. This adresses *Biaised* spectators opinions on tournament quality as it explores the performances of a competition design regarding the a priori seedings.

### How to Use This Dashboard:

Each section contains **interactive widgets**:
- **System Checkboxes**: Select which tournament variants to compare
- **Solver Checkboxes**: Choose game outcome generator (BetterWin/Logistic/MinEffort)
- **Population Checkboxes**: Choose skill distribution model
- **Seeding Checkboxes**: Choose initial seeds quality

**Tip:** Start by comparing the **Original** (problematic Twitter example), **Major** (current Valve implementation), and **Ema** (proposed solution) across different conditions. For perspective, add the alternative sport system **4groups4**.


## Systems Being Compared

- **2Groups8**: Baseline - Two groups of 8 teams (Round Robin)
- **4Groups4**: Baseline - Four groups of 4 teams (Round Robin)
- **Original**: The matching system from the Twitter example (F1_CROSS with tiebreaker)
- **Original-NTB**: Same as Original but without tiebreaker
- **Major**: Current Valve CS Major implementation (speed-up matching)
- **Major-NTB**: Same as Major but without tiebreaker
- **FixedDefault**: Simple fix using Chord6 for Round 5
- **Ema**: Expected Matching Approach using F1_ROT algorithm

## Understanding the Data

- **20 simulation samples** per configuration
- **Configuration** consist of a (Solver, level model, Outcome generator, Initial Seeding)
- **Fair Comparaison** In a given configuration, for two different systems, the outcome of an encounter *teamA versus teamB* is identical.
- **Results** are aggregated and displayed with mean on the x-axis and variance on the y-axis.
- **Perspective and Truth** Each metric uses as input a Ranking, the plot on the left uses **True Level Ranking**, the right side is the **Seeding**. True level represent what competitors experience, the seedings represent the perspective spectators have on competition.
- **Interactive** - change parameters to see how results shift

## Pitfall

It does not make much sense to compare SwissBracket and Group stage with Fairness or Balance metrics. By design, teams do not have the same ability to participate in a given round.

## 1. Utility Loss

### What is Utility Loss?

**Utility Loss** measures how much "value" we lose by qualifying the wrong teams. It compares the actual qualified teams against the optimal qualification. unlike top-8 precision, it nuances the qualification of a team by its level of competitivness.

### How It's Calculated:

For each rank, we have a **baseline qualification probability** (from Round Robin simulations - see Notebook 2, section 2.2). This represents how likely each corresponding team would be to qualify in a "perfectly fair" format - unbiased by seedings or pairing.

**Utility Score:**
- Actual qualified teams' baseline probabilities are summed
- This is compared to the theoretical maximum (if the true top 8 always qualified)

**Utility Loss = Maximum Possible Score - Actual Score**

### Interpretation:

- **Lower is better** (less value lost)
- **0 = Perfect**: Always qualifies the teams most likely to deserve it
- **High values**: Frequently qualifies teams that shouldn't make it

In [2]:
from visualisation.dashboard import DashboardVisualizer

dashboard = DashboardVisualizer(samples)
dashboard.plot_utility_loss_robustness()

## 2. Top 8 Precision

### What is Top 8 Precision?

**Precision** measures what percentage of the qualified teams are actually from the ranking top 8.

### How It's Calculated:

```
Precision = (Number of true top-8 teams that qualified) / 8
```

**Example:**
- If 6 of the true top-8 teams qualify, precision = 6/8 = 0.75 (75%)
- Perfect system would have precision = 1.0 (100%)

### Interpretation:

- **Higher is better** (closer to 1.0)
- **1.0 = Perfect**: All 8 qualified teams are from the true top 8
- **0.5 = Random**: No better than flipping a coin


In [3]:
dashboard.plot_top8_precision()

## 3. Fairness

### What is Fairness?

**Fairness** measures how well do the best teams perform during the tournament.

### The Fairness Concept:

High rank must be winner. This is achieved by giving top teams weak opponents as often as possible.

**Unfair scenarios:**
- Seed 1 vs Seed 2 when both are 2-1 (lose for a high rank)
- Seed 15 vs Seed 16 when both are 1-2 (win for a low rank)

### How It's Calculated:

$\frac{1}{N}\sum_{i=1}^{N} r_{w,i}$

Where N is the number of games, and $r_{w,i}$ the rank of the winner of game i.

### Interpretation:

- **Lower is better** High ranks win a lot of games
- **Perfect Matching** on $K_{3,3}$ are optimal regarding fairness
- **Perfect fairness**: Undefined - see below.
- **Poor fairness**: Frequently pairs top vs top or bottom vs bottom

Fairness is a metric from game theory and is defined over the perfect seed & BetterWin condition - just like the twitter example.

It present practical challenges for interpretation:
- **Late Top seed qualification** You can increase the fairness by qualifying top teams in this order: 7,8 in $Round_3$; 5,6,7 in $Round_4$ and finaly rank 1,2,3 at the very end. Is this possible? With the MinEffort Solver - maybe.
- Does not allow comparaison with **4groups4** and **8groups8** due to a core differences: in Swiss Bracket teams are not guaranteed to play the same amount of games.


### Relevance:
It suites the twitter examples that resulted in a rule change. It directly aims and adresses the discussion of the threads and does highlight the theoretical supperiority of the current implementation versus the original system.



In [4]:
dashboard.plot_fairness()

## 4. Balance

### What is Balance?

**Balance** measures how **competitive** the games are - i.e., how closely matched the teams are.

### The Balance Concept:

- **Balanced game**: Two teams of similar rank face each other (could go either way). It adds entertainement values
- **Imbalanced game**: Strong team vs weak team (outcome predictable). Boring match.

**Example:**
- Balanced: rank 8 vs rank 9 (both mid-tier)
- Imbalanced: True rank 1 vs True rank 16 (obvious mismatch)

### How It's Calculated:

For each game, we compute the **true skill difference** between the two teams (based on their actual levels, not seeds).

**Balance Score = Average absolute skill difference across all games**

### Interpretation:

- **Lower is better** smaller skill gaps = closer games
- **Perfect balance**: Every game is 50-50
- **Poor balance**: Many blowouts, few competitive games

### Relevance

- **Problematic pairings** in the last round are more Balanced than perfect matching

### Balance vs Fairness:

The two notion are related. From a formal perspective, pairing systems are trade-off between the two notions.

**Key Distinction:**
- **Fairness** Is what Elmination Bracket aim at.
- **Balance** Is what Swiss Rounds aim at.

Swiss Bracket is a mix of the two. 

In [5]:
dashboard.plot_balance()


## 5. Last Round Metrics

The original example contains a single balance/fairness issue in the very last round.

In that regard, we plot the metrics computed on the last three games of the tournament.


### 5.1 Last Round Fairness

In [6]:
dashboard.plot_last_round_fairness()

### 5.2 Balance

In [7]:
dashboard.plot_last_round_balance()

### 5.3 Last-Round Matching Quality

Round 5 is the decisive round: all 6 remaining 2-2 teams must be paired for 3 qualification matches. Valve's rulebook defines an ordered preference list of 15 possible matchings—ranked from most to least preferred. **Matching priority 1--6** correspond to the six *preferred* matchings: perfect matchings on {3,3}$ that pair top-half seeds (1--8) against bottom-half seeds (9--16), respecting seeding structure. Priorities 7--15 are non-preferred and involve at least one top-vs-top or bottom-vs-bottom pairing.

The heatmap below shows the distribution of matching priorities used in Round 5 across all simulated tournaments, for each system. Each row is a system; each column is a priority (1--15); colour intensity encodes frequency. Darker cells at **low priority indices** indicate a system reliably uses preferred pairings; spread toward high indices signals structural failures.

**Key observations:**
- From the **level perspective** (true skill), EMA and no-tiebreaker variants select a top-6 preferred pairing in ~54% of Round-5 games, compared to ~42% for Original and Major---barely above the 40% random baseline (6 out of 15 matchings).
- From the **seed perspective** (initial seedings), the difference is dramatic: EMA achieves **~95% top-6 preferred** versus only ~36--38% for Original and Major. This reflects EMA's structural guarantee: when the seedings are respected, the rotation algorithm always lands in the preferred coloring class, while tiebreaker-based systems disrupt this structure.

In [8]:
dashboard.plot_last_round_matching_heatmap()

### 5.4 Top-6 Preferred Pairing Rate

The bar chart below summarises the heatmap above as a single number: the percentage of Round-5 games in which a system selected one of the six preferred {3,3}$ matchings (priorities 1--6). The red dashed reference line at **40%** represents the random baseline (6 preferred out of 15 total matchings).

Systems that consistently exceed this baseline demonstrate that their pairing algorithm is structurally aligned with the preferred coloring class, rather than falling back on non-preferred matchings due to rematch constraints or tiebreaker interference. The seed perspective (right panel) most clearly separates EMA and no-tiebreaker variants (~95%) from the current Major system (~38%), confirming that EMA's rotation-based construction is the primary driver of last-round matching quality.

In [9]:
dashboard.plot_top6_preferred_pairings()

## 6. Frustration

### What is Frustration?

**Frustration** captures scenarios where the tournament produces qualifications that feel **unfair** or **wrong** to participants and observers.

### Frustration Concept:

**Head-to-Head Violations**
- Team A beats Team B directly
- Team B qualifies, Team A doesn't
- *"We beat them, how did they advance over us?"*

This is refered as **Head-to-Head Violations**


### How It's Calculated:

Number of Head-to-Head Violations per tournament

### Interpretation:

- **Lower is better** (fewer frustrating scenarios)
- **0 = Ideal**: No problematic outcomes
- **High values**: Format regularly produces debatable qualification.

### Why Frustration Matters:

The metric is an attempt from my side to rational complains the system receives. Specificly, it adresses the *counter example* I proposed in [section 1.2.3 limitation](2_Simulation.ipynb#123-Limitations).

I assume it:
- Damage tournament credibility
- Create controversy and complaints

However I do not think it undermines competitive integrity


In [10]:
dashboard.plot_frustration()